In [0]:
%sql
SHOW CATALOGS;

catalog
adb_salesananlytics
catalog
samples
system


In [0]:

storage_path = (
    "abfss://ct-salesdata@"
    "strsalesdata1.dfs.core.windows.net/"
)

# List files in Azure Storage
display(dbutils.fs.ls(storage_path))


path,name,size,modificationTime
abfss://ct-salesdata@strsalesdata1.dfs.core.windows.net/invalid/,invalid/,0,0
abfss://ct-salesdata@strsalesdata1.dfs.core.windows.net/processed/,processed/,0,0
abfss://ct-salesdata@strsalesdata1.dfs.core.windows.net/raw/,raw/,0,0
abfss://ct-salesdata@strsalesdata1.dfs.core.windows.net/valid/,valid/,0,0


In [0]:
df= spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/salesanalytics_project/salesanalytics_schema/sales_volume/sales (1).csv")
display(df)

sale_id,sale_date,customer_id,product_id,quantity,unit_price
50001,9/1/2026,1001,2001,1,1299.99
50002,9/1/2026,1002,2002,2,29.99
50003,9/2/2026,1003,2004,1,399.99
50004,9/2/2026,1004,2006,1,249.99
50005,9/3/2026,1005,2008,1,299.99
50006,9/3/2026,1006,2003,1,89.99
50007,9/4/2026,1007,2009,1,449.99
50008,9/4/2026,1008,2010,2,119.99
50009,9/5/2026,1009,2005,1,149.99
50010,9/5/2026,1010,2007,2,79.99


In [0]:
# Write the DataFrame to a Delta table named 'Orders' in the designated schema
df.write.format("delta").mode("overwrite").saveAsTable("salesanalytics_project.salesanalytics_schema.Orders")

In [0]:
%sql
DESCRIBE EXTENDED salesanalytics_project.salesanalytics_schema.Orders

col_name,data_type,comment
sale_id,int,null
sale_date,string,null
customer_id,int,null
product_id,int,null
quantity,int,null
unit_price,double,null
# Clustering Information,,
# col_name,data_type,comment
,,
# Delta Statistics Columns,,


In [0]:
from pyspark.sql.functions import to_date, col
df = df.withColumn("sale_date", to_date(col("sale_date"), "M/d/yyyy"))

In [0]:
df=df.withColumn("total_amount",col("quantity")*col("unit_price"))

In [0]:
df_valid=df.filter(col("quantity")>0)
df_invalid=df.filter(col("quantity")<=0)

In [0]:
print("Total records", df.count())
print("Valid records", df_valid.count())
print("Invalid records", df_invalid.count())

Total records 20
Valid records 20
Invalid records 0


In [0]:
df_valid.write\
    .mode("delta").mode("overwrite")\
    .option("header", True)\
    .csv("abfss://ct-salesdata@strsalesdata1.dfs.core.windows.net/processed/valid/valid_sales")


In [0]:
df_invalid.write\
    .mode("delta").mode("overwrite")\
    .option("header", True)\
    .csv("abfss://ct-salesdata@strsalesdata1.dfs.core.windows.net/processed/invalid/invalid_sales")

In [0]:
df_valid.write\
    .mode("overwrite")\
    .format("delta")\
    .option("header", True)\
    .csv("abfss://ct-salesdata@strsalesdata1.dfs.core.windows.net/processed/delta/valid_sales")